# Medir el error de un modelo de regresión

Todos los notebooks hasta ahora han resumido los errores de un modelo de regresión en un solo número, normalmente el error absoluto medio de la [regresión lineal](../algorithms/linear-models/linear_regression.ipynb). Un número es necesario para comparar modelos, pero hay varios entre los que elegir, y cada uno responde a una pregunta distinta: cuánto se desvían las predicciones de media, lo malas que son las peores, cuánto mejor que no saber nada, cuánto se desvían en proporción al valor. Elegir uno es elegir qué pregunta importa.

## Un modelo que medir

El modelo de gradient boosting de [Gradient boosting](../algorithms/tree-based/gradient_boosting.ipynb), con las viviendas de California, entrenado con las filas de entrenamiento y medido con las filas de prueba. Los valores se convierten a dólares:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
y_predicted = model.predict(X_test)

## MAE y RMSE

El **error absoluto medio** es la distancia media entre la predicción y el valor real. La **raíz del error cuadrático medio** eleva cada error al cuadrado, promedia y saca la raíz para volver a las unidades del objetivo:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i| \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

{"MAE ($)": mean_absolute_error(y_test, y_predicted), "RMSE ($)": root_mean_squared_error(y_test, y_predicted)}

{'MAE ($)': 31105.521289073844, 'RMSE ($)': 46539.919537971495}

Los dos están en dólares, pero difieren, porque elevar al cuadrado da más peso a los errores grandes. Dos conjuntos de predicciones para las mismas cinco casas, en miles de dólares, hacen visible la diferencia: en el primero, todas las predicciones se desvían 10; en el segundo, cuatro son exactas y una se desvía 50:

In [ ]:
import numpy as np
import pandas as pd

true_values = np.array([200, 150, 300, 250, 180])
pd.DataFrame({
    name: {"MAE": mean_absolute_error(true_values, true_values + errors), "RMSE": root_mean_squared_error(true_values, true_values + errors)}
    for name, errors in {
        "every one off by 10": np.array([10, -10, 10, -10, 10]),
        "one off by 50": np.array([0, 0, 0, 0, 50]),
    }.items()
}).T.round(1)

,MAE,RMSE
every one off by 10,10.0,10.0
one off by 50,10.0,22.4


El mismo error total da el mismo MAE, pero el RMSE del segundo es más del doble: al cuadrado, un solo error de 50 pesa tanto como 25 errores de 10. El RMSE nunca es menor que el MAE, y cuanto más se separan, más se concentra el error en unos pocos fallos grandes. En el modelo de arriba, un RMSE bastante por encima del MAE dice lo mismo: la mayoría de los distritos se predicen bastante bien, y unos pocos muy mal.

Cuál comunicar depende de lo que cueste un error grande. Si un error de 50 000 \$ es peor que cinco de 10 000 \$, lo dice el RMSE; si son igual de malos, el MAE. El MAE también se ve menos afectado por unos pocos valores atípicos en el conjunto de prueba.

## El error que minimiza un modelo, y el error con el que se le juzga

La función que minimiza un modelo mientras aprende es su **pérdida** (*loss*); la medida con la que se le juzga después es su **métrica**. En regresión suelen ser de la misma familia: la regresión lineal minimiza el error cuadrático, y a menudo se la juzga por el RMSE. No tiene por qué ser así. Una pérdida debe adaptarse al algoritmo de aprendizaje (los métodos basados en gradientes necesitan que tenga derivada), mientras que una métrica debe adaptarse a la pregunta que se hace. En clasificación las dos suelen ser distintas: un modelo aprende minimizando la pérdida logarítmica, y se le juzga por su precisión o su exhaustividad, que no se pueden optimizar directamente: cambian a saltos, sin una derivada útil ([Evaluar un clasificador](classification_metrics.ipynb)).

## Una referencia con la que comparar

Un error de 30 000 \$ solo es bueno o malo en comparación. La comparación más sencilla es un modelo que ignora las variables y predice el mismo valor para todos los distritos: `DummyRegressor`. ¿Qué constante? La media de los objetivos de entrenamiento es la constante con el menor error cuadrático; la mediana es la constante con el menor error absoluto. Las dos, con las dos medidas:

In [ ]:
from sklearn.dummy import DummyRegressor

baselines = {
    "always the mean": DummyRegressor(strategy="mean").fit(X_train, y_train),
    "always the median": DummyRegressor(strategy="median").fit(X_train, y_train),
}
pd.DataFrame({
    name: {"MAE ($)": mean_absolute_error(y_test, baseline.predict(X_test)),
           "RMSE ($)": root_mean_squared_error(y_test, baseline.predict(X_test))}
    for name, baseline in baselines.items()
}).T.round(0)

,MAE ($),RMSE ($)
always the mean,90607.0,114486.0
always the median,87404.0,117312.0


Predecir el mismo valor para todos los distritos falla por unos 90 000 \$ de media, y el modelo por unos 31 000 \$: elimina dos tercios del error de no saber nada. Cada constante gana con la medida que minimiza: la mediana tiene el MAE más bajo (87 400 \$ frente a 90 600 \$), y la media el RMSE más bajo. Incluso una referencia hay que elegirla según la medida con la que se va a juzgar.

## R²

El **coeficiente de determinación**, R², incorpora a la medida la comparación con la media:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

la proporción de la varianza del objetivo que explica el modelo: 1 para un modelo perfecto, 0 para uno que siempre predice la media $\bar{y}$ de los datos con los que se mide. Con datos nuevos no tiene límite inferior, y un modelo peor que la media tiene un R² negativo. El modelo, la media, y las predicciones del modelo desplazadas 50 000 \$:

In [ ]:
from sklearn.metrics import r2_score

{
    "model": r2_score(y_test, y_predicted),
    "always the training mean": r2_score(y_test, baselines["always the mean"].predict(X_test)),
    "model, shifted by $50,000": r2_score(y_test, y_predicted + 50_000),
    "model, shifted by $150,000": r2_score(y_test, y_predicted + 150_000),
}

{'model': 0.8347108310026643,
 'always the training mean': -0.00021908714592466794,
 'model, shifted by $50,000': 0.6379795047054088,
 'model, shifted by $150,000': -0.9001639497663978}

El modelo explica alrededor del 83 % de la varianza de los valores de prueba. La media de entrenamiento puntúa casi exactamente 0, ligeramente por debajo, porque la media propia de las filas de prueba difiere un poco de la de las de entrenamiento. Desplazar todas las predicciones 50 000 \$ baja el R² a 0,64, y 150 000 \$ a -0,90: un modelo que se desvía sistemáticamente tanto es peor que predecir la media, y el R² lo dice con un número negativo. El R² también depende de los datos con los que se mide: medido solo con distritos de valor parecido, el mismo modelo obtendría un R² más bajo, porque habría menos varianza que explicar.

## Errores en proporción al valor

Un fallo de 30 000 \$ es pequeño en un distrito que vale 500 000 \$ y enorme en uno que vale 60 000 \$. Los errores absolutos dan más peso a los distritos caros; para muchos usos, entre ellos la tasación inmobiliaria, lo que importa es el error en proporción al valor. El **error porcentual absoluto** de cada predicción es $|y_i - \hat{y}_i| / |y_i|$, y de él salen tres resúmenes habituales:

- **MAPE**, su media;
- **MdAPE**, su mediana, menos afectada por unos pocos porcentajes enormes;
- la **proporción de predicciones dentro del 10 %** del valor real (a veces llamada PE10), la cifra habitual de los modelos de tasación automática: qué proporción de las tasaciones son fiables con un margen del 10 %.

In [ ]:
def percentage_measures(y_true, y_pred):
    percentage_errors = np.abs(y_true - y_pred) / np.abs(y_true)
    return {
        "MAPE (%)": 100 * percentage_errors.mean(),
        "MdAPE (%)": 100 * np.median(percentage_errors),
        "within 10% (%)": 100 * (percentage_errors <= 0.10).mean(),
        "within 20% (%)": 100 * (percentage_errors <= 0.20).mean(),
    }


pd.DataFrame({
    "model": percentage_measures(y_test, y_predicted),
    "always the median": percentage_measures(y_test, baselines["always the median"].predict(X_test)),
}).T.round(1)

,MAPE (%),MdAPE (%),within 10% (%),within 20% (%)
model,18.0,11.8,43.7,70.4
always the median,53.1,36.2,14.8,28.5


De media el modelo se desvía un 18 % del valor, pero la mitad de sus predicciones están dentro del 11,8 %: la media la suben una minoría de porcentajes grandes. El 44 % de los distritos se tasan dentro del 10 %, y el 70 % dentro del 20 %; la referencia de la mediana consigue un 15 % dentro del 10 %. De todos estos números, «dentro del 10 %» es el más fácil de explicar a quien vaya a usar las predicciones: algo menos de la mitad de estas tasaciones son fiables con ese margen.

Los errores porcentuales tienen dos trampas. No están definidos cuando el valor real es 0, y se disparan cuando está cerca de 0, así que encajan con objetivos que siempre están muy por encima, como los precios. Y son asimétricos: una predicción por debajo nunca puede desviarse más del 100 % (mientras ninguna predicción sea negativa), mientras que una por encima puede desviarse cualquier cantidad, así que un modelo entrenado para minimizar el MAPE aprende a predecir bajo.

## Los mismos errores, por precio

Dividir los distritos de prueba en cinco tramos de valor real muestra cómo los dos tipos de medida cuentan historias distintas:

In [ ]:
errors = pd.DataFrame({"true value": y_test, "absolute error": np.abs(y_test - y_predicted)})
errors["percentage error"] = 100 * errors["absolute error"] / errors["true value"]
errors["price band"] = pd.qcut(errors["true value"], 5, labels=["cheapest fifth", "second", "third", "fourth", "most expensive fifth"])
errors.groupby("price band", observed=True)[["absolute error", "percentage error"]].agg(["mean", "median"]).round(0)

absolute error          percentage error       
                               mean   median             mean median
price band                                                          
cheapest fifth              21390.0  15290.0             32.0   20.0
second                      23308.0  16087.0             18.0   13.0
third                       24126.0  17841.0             14.0   10.0
fourth                      32749.0  23662.0             13.0   10.0
most expensive fifth        53949.0  38361.0             14.0   10.0

El error absoluto crece con el precio: unos 21 000 \$ de media para la quinta parte más barata de los distritos, y 54 000 \$ para la más cara. El error porcentual va en sentido contrario: 32 % para la quinta parte más barata, 18 % para la segunda, y entre el 13 % y el 14 % para el resto. Juzgado en dólares, el problema del modelo son los distritos caros; juzgado en proporción, los baratos, donde un error de 20 000 \$ es un tercio del valor. Ninguna visión es incorrecta, y el uso de las predicciones decide cuál importa. Parte del error de la quinta parte más cara viene del tope de 500 000 \$ visto en [Viviendas de California, paso a paso](../basic-workflows/california_by_hand.ipynb).

## Qué medida responde a qué pregunta

| Pregunta | Medida |
| --- | --- |
| ¿Cuánto se desvía una predicción típica, en las unidades del objetivo? | MAE |
| ¿Cuánto se desvía, contando los fallos grandes como peores que varios pequeños? | RMSE |
| ¿Cuánto mejor que no saber nada? | la misma medida para una referencia; R² frente a la media |
| ¿Cuánto se desvía en proporción al valor? | MAPE, o MdAPE si unos pocos valores pequeños lo disparan |
| ¿Cuántas predicciones son fiables dentro de un margen? | la proporción dentro del 10 % (o del 5 %, o del 20 %) |
| ¿Dónde falla el modelo? | cualquiera de ellas, por grupo: [Análisis de errores: dónde falla un modelo](error_analysis.ipynb) |

Un informe da más de una, y siempre junto a una referencia. La medida que decide entre modelos hay que elegirla antes de compararlos, para que la elección no la haga la medida que favorezca al modelo que uno ya prefiere.